# Newtonian Physics Through Numerical Integration

### Euler's method applied to three classical force problems

**Abstract.** Many mechanics problems can be written as first order differential equations for position and velocity. This article develops the explicit Euler method and applies the same numerical procedure to three different systems: motion with quadratic air resistance, a gravitational orbit, and Rutherford scattering. The aim is to make the connection between a physical force law and its numerical implementation transparent.

## The numerical method

Newton's second law may be written as

$$\dot{\mathbf r}=\mathbf v, \qquad \dot{\mathbf v}=\mathbf a(\mathbf r,\mathbf v,t).$$

For a small time step $\Delta t$, the explicit Euler method gives

$$\boxed{\mathbf v_{n+1}=\mathbf v_n+\mathbf a_n\Delta t}, \qquad \boxed{\mathbf r_{n+1}=\mathbf r_n+\mathbf v_n\Delta t}.$$

Thus the computational loop is simply: **initial conditions → acceleration → velocity update → position update → repeat**. The force law changes from problem to problem; the numerical machinery does not.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams.update({'font.size': 11, 'axes.spines.top': False, 'axes.spines.right': False})

## Motion with air resistance

This first example is one-dimensional, so we work with the vertical acceleration component $a_y$ rather than a two-dimensional acceleration vector. Taking upward as positive, gravity contributes $-g$. For quadratic drag,

$$\boxed{a_y=-g-cv\lvert v\rvert}.
$$

The factor $v\lvert v\rvert$ ensures that the drag force always opposes the direction of motion.

In [ ]:
dt, T = 0.05, 8.0
g, c = 9.81, 0.018
t = np.arange(0, T + dt, dt)
h = np.empty_like(t); v = np.empty_like(t); a_y = np.empty_like(t)
h[0], v[0] = 2.5, 0.0
for i in range(len(t)-1):
    a_y[i] = -g - c*v[i]*abs(v[i])
    v[i+1] = v[i] + a_y[i]*dt
    h[i+1] = h[i] + v[i]*dt
a_y[-1] = -g - c*v[-1]*abs(v[-1])
fig, ax = plt.subplots(3, 1, figsize=(7, 7), sharex=True)
ax[0].plot(t,h); ax[0].set_ylabel('height')
ax[1].plot(t,v); ax[1].set_ylabel('velocity')
ax[2].plot(t,a_y); ax[2].set_ylabel('vertical acceleration'); ax[2].set_xlabel('time')
fig.suptitle('Falling object with quadratic air resistance'); fig.tight_layout(); plt.show()

Because this model has only one spatial coordinate, the scalar $a_y$ is sufficient. In the next two examples the particle moves in the $x$-$y$ plane, so the acceleration is written as the vector $\mathbf a$.

## Gravitational orbit

For a particle moving around a fixed mass $M$,

$$\mathbf a=-\frac{GM}{r^3}\mathbf r.$$

The acceleration points toward the centre and changes direction continuously as the particle moves.

In [ ]:
G, M = 1.0, 1.0
dt, T = 0.002, 20.0
N = int(T/dt)
x = np.empty(N+1); y = np.empty(N+1); vx = np.empty(N+1); vy = np.empty(N+1)
x[0], y[0] = 1.0, 0.0; vx[0], vy[0] = 0.0, 0.85
for i in range(N):
    r = np.hypot(x[i], y[i])
    ax = -G*M*x[i]/r**3; ay = -G*M*y[i]/r**3
    vx[i+1] = vx[i] + ax*dt; vy[i+1] = vy[i] + ay*dt
    x[i+1] = x[i] + vx[i]*dt; y[i+1] = y[i] + vy[i]*dt
plt.figure(figsize=(6.5,6.5)); plt.plot(x,y,lw=1.5,label='particle')
plt.scatter([0],[0],s=70,label='central mass'); plt.xlabel('x'); plt.ylabel('y')
plt.title('Euler integration of a gravitational orbit'); plt.axis('equal'); plt.grid(alpha=0.25); plt.legend(); plt.show()

## Rutherford scattering

For a repulsive Coulomb interaction, using convenient units,

$$\mathbf a=+CQ\frac{\mathbf r}{r^3}.$$

The positive sign means that the acceleration points away from the target. A projectile with impact parameter $b$ is therefore deflected as it passes the nucleus.

In [ ]:
C, Q = 1.0, 2.0; b = 2.5; x0 = -12.0; speed0 = 4.0
dt, T = 0.001, 7.0; N = int(T/dt)
x = np.empty(N+1); y = np.empty(N+1); vx = np.empty(N+1); vy = np.empty(N+1)
x[0], y[0] = x0, b; vx[0], vy[0] = speed0, 0.0
for i in range(N):
    r2 = x[i]**2 + y[i]**2; r3 = r2**1.5
    ax = C*Q*x[i]/r3; ay = C*Q*y[i]/r3
    vx[i+1] = vx[i] + ax*dt; vy[i+1] = vy[i] + ay*dt
    x[i+1] = x[i] + vx[i]*dt; y[i+1] = y[i] + vy[i]*dt
angle = np.degrees(np.arctan2(vy[-1],vx[-1])); print(f'Estimated outgoing angle: {angle:.3f} degrees')
plt.figure(figsize=(8,5)); plt.plot(x,y,lw=1.5,label=f'trajectory, b={b}')
plt.scatter([0],[0],s=70,label='target nucleus'); plt.xlabel('x'); plt.ylabel('y')
plt.title('Numerical Rutherford scattering'); plt.grid(alpha=0.25); plt.legend(); plt.show()

## What the three examples have in common

The three systems have very different physical force laws, but they all have the same numerical structure.

| **System** | **Acceleration** | **Qualitative behaviour** |
|:---|:---|:---|
| Air resistance | $a_y=-g-cv\lvert v\rvert$ | Damped / falling motion |
| Gravity | $\mathbf a=-GM\mathbf r/r^3$ | Bound orbit |
| Coulomb repulsion | $\mathbf a=+CQ\mathbf r/r^3$ | Scattering |

**Table 1.** Comparison of the three force laws. The dimensional form of the acceleration depends on the problem: the drag model is one-dimensional, while the gravitational and Coulomb models are two-dimensional.

The numerical algorithm is identical in all three cases:

1. Start from the initial position and velocity.
2. Evaluate the appropriate acceleration (or acceleration component).
3. Update velocity using Euler's method.
4. Update position using Euler's method.
5. Advance time and repeat.

The essential chain is therefore

$$\boxed{\text{force law}\;\longrightarrow\;\text{acceleration}\;\longrightarrow\;\text{velocity}\;\longrightarrow\;\text{position}}.$$

## Accuracy and timestep

Euler's method is first order, so its global error scales as $O(\Delta t)$. A smaller timestep generally improves the approximation, but requires more calculations. This is especially important when the acceleration varies rapidly, as in close gravitational encounters or scattering.

For high accuracy work, methods such as Runge–Kutta or symplectic integrators are preferable. Euler's method remains valuable because the relationship between the differential equation and the numerical algorithm is exceptionally transparent.

## Conclusion

A falling object, a gravitational orbit, and a scattering event may appear unrelated. Numerically, however, they can all be expressed as a position equation together with a velocity equation.

$$\dot{\mathbf r}=\mathbf v,\qquad \dot{\mathbf v}=\mathbf a(\mathbf r,\mathbf v).$$

For the one-dimensional drag problem, this reduces to the corresponding scalar vertical component $a_y$. Euler's method then converts these equations into a sequence of elementary updates. The force law determines the physical behaviour; the integrator provides the common computational framework.